# Annotate movement measurements from selected trials

Load real recordings with the same workflow as the data template, calculate speed visibly with movement, and pass each selected trial to the existing annotation/plot functions. Each trial gets a separate axis so omitted intervals are never joined.

In [ ]:
# === 1| Find the Existing Source Checkout =======================================

from pathlib import Path
import sys

source_root = None
for parent in (Path.cwd(), *Path.cwd().parents):
    for candidate in (parent, parent / 'src'):
        if (candidate / 'data_conduit').is_dir() and (candidate / 'movement_figures').is_dir():
            source_root = candidate                      # Both projects must come from the same edited checkout.
            break
    if source_root is not None:
        break

if source_root is None:
    raise FileNotFoundError('Open this notebook inside the checkout containing both projects.')
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))                  # Import these local source files without installing or rewriting anything.

# === 2| Import the Existing DataStructure and Its Analysis Operations ============

import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

from data_conduit.datastructures import slice_stream, slice_stream_for_trial
from data_conduit.integrations.DLC.pose import pose_to_movement
from data_conduit.refactor_qc.training_filter import filter_trials, training_spec
from movement_figures.data_template.loading import build_datastructure, prepare_pose
from movement_figures.video import read_session_video_frame

import matplotlib.pyplot as plt
from movement.kinematics import compute_speed
from movement_figures.timeseries_template.annotations import Annotations, qc_annotations, draw_annotations

## Choose the actual recording inputs

The settings have the same meaning as the data template. `TRACKING_KEYPOINT` must be a keypoint present in your DLC output.

In [ ]:
# === 1| Select Mice, Days and Recording Folders Before Reading Files =============

ROOT = Path('/path/to/BonsaiOutput/Training')             # Replace with the directory containing your actual recordings.
LEVEL_NAMES = ('mouseID', 'day')                         # Use () if ROOT directly contains recording folders.
LEVEL_SELECTORS = {}                                    # Example choices belong here: l0_selector and l1_selector.
INCLUDE_SESSIONS = None                                 # Optional list of recording folder names; None keeps selected folders.
STREAMS = ('trials', 'events', 'video', 'dlc')             # These figures require events/trials and aligned pose; other sources remain available.

# === 2| Select Trial Rows Before Passing Data to Plotting Functions ==============

TRIAL_RANGE = (1, 8)                                    # Inclusive original trial numbers, applied independently to every recording.
TRAINING_RULES = None                                   # Set training_spec() to reuse refactor_qc's session/min-trial/LED rules.
EXCLUDE_LED_ON = False                                  # True additionally removes all ON-labelled trial rows from the selection.

# === 3| Choose Optional Pose Processing Explicitly ==============================

CONFIDENCE_THRESHOLD = None                             # None preserves raw coordinates; set a threshold after checking DLC likelihoods.
MAX_GAP_FRAMES = None                                   # None keeps tracking gaps; a positive integer fills only short internal runs.
SMOOTHING_WINDOW = None                                 # None disables smoothing; otherwise use an odd frame count of at least three.
READ_BACKGROUND = False                                 # True decodes the first video frame for each selected recording.

TRACKING_KEYPOINT = 'body'                               # Use the actual body keypoint in these recordings.

## Read and select data before drawing annotations

The existing training filter is optional. It runs before the explicit trial-range selection. Pose processing remains separate for each recording.

In [ ]:
# === 1| Build the Existing DataStructure and Preview the Selected Directories =====

datastructure = build_datastructure(
    ROOT,
    level_names=LEVEL_NAMES,
    streams=STREAMS,
    include=INCLUDE_SESSIONS,
    **LEVEL_SELECTORS,
)

selected_sessions = datastructure.select()              # Directory selection only; no experimental data have been read yet.
display(pd.DataFrame([
    {'session': key, 'path': str(ref.path), **ref.levels}
    for key, ref in selected_sessions.items()
]))


# === 1| Read the Selected Recordings and Inspect the Plain Trial Table ===========

streams = datastructure.load()                          # Returns the existing StreamMap, a mapping of pandas/xarray objects.
trials = streams['trials']
display(trials.head())

# === 2| Optionally Reuse the Existing Training Session and Trial Rules ============

selected_trials = trials.copy()
if TRAINING_RULES is not None:
    folder_names = {key: ref.path.name for key, ref in datastructure.sessions.items()}
    selected_trials['session_name'] = selected_trials['session'].map(folder_names)
    rules = TRAINING_RULES.rename(columns={'session': 'session_name'})
    selected_trials = filter_trials(
        selected_trials,
        rules,
        session_column='session_name',                  # Spreadsheet folder names are distinct from collision-aware stream identities.
        report=False,
    )                                                   # Keeps the stable session column for subsequent pose/event selection.

# === 3| Apply the Requested Original Trial Range Within Every Recording ==========

selected_trials = slice_stream(
    selected_trials,
    selectors={'trial_index': slice(*TRIAL_RANGE)},                # Each recording numbers its own trials; this does not select global row positions.
)
if EXCLUDE_LED_ON:
    selected_trials = slice_stream(selected_trials, where=lambda rows: rows['LED'].ne('ON'))

display(selected_trials)


# === 1| Convert and Process Full Recordings Separately ===========================

raw_pose_by_session = {}
pose_by_session = {}
backgrounds = {}

for session_id in selected_trials['session'].drop_duplicates():
    if 'dlc:position' not in streams or 'dlc:confidence' not in streams:
        print('No DLC pose was loaded; trial/event tables remain accessible.')
        break

    position = slice_stream(streams['dlc:position'], selectors={'session': session_id})
    confidence = slice_stream(streams['dlc:confidence'], selectors={'session': session_id})
    if position.sizes['Time'] == 0:
        print(f'{session_id}: no DLC pose available; trial data remain accessible.')
        continue

    raw_pose = pose_to_movement(position, confidence)     # Existing converter; acquired seconds and recording identity are retained.
    pose = prepare_pose(
        raw_pose,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        max_gap_frames=MAX_GAP_FRAMES,
        smoothing_window=SMOOTHING_WINDOW,
    )                                                   # Full-recording processing avoids artificial edges at selected trial boundaries.

    raw_pose_by_session[session_id] = raw_pose
    pose_by_session[session_id] = pose
    if READ_BACKGROUND:
        frame, video_path = read_session_video_frame(datastructure.sessions[session_id].path)
        backgrounds[session_id] = (frame, video_path)   # Each background belongs to this recording's camera coordinates.

# === 2| Compare Missing Coordinates Without Modifying Either Dataset =============

display(pd.DataFrame([
    {
        'session': key,
        'raw_missing': int(raw_pose_by_session[key].position.isnull().sum()),
        'processed_missing': int(pose.position.isnull().sum()),
    }
    for key, pose in pose_by_session.items()
]))

## Keep the existing explicit plotting function

This function receives only the selected time values, measured signal, annotation records and display settings. Its original implementation is retained.

In [ ]:
def plot_annotated_trace(
    time,
    values,
    annotations: Annotations,
    *,
    window: tuple[float, float],
    time_offset: float = 0.0,
    ylabel: str = "Amplitude",
    title: str = "Annotated time series",
    ylim: tuple[float, float] | None = None,
    trace_label: str | None = None,
):
    """Return a figure, axis and annotation result for one supplied time series.

    Parameters
    ----------
    time, values : one-dimensional array-like
        Matching samples; time is absolute session seconds. Missing values
        remain gaps. Annotation records use that same absolute clock.
    annotations : Annotations
        Explicit regions and events, typically returned by qc_annotations.
    window : (start, end)
        Absolute interval to display. Subtract time_offset only for presentation.
    time_offset : float
        Origin displayed as zero; does not alter input arrays or annotations.
    ylabel, title, ylim, trace_label
        Axis text, optional limits and optional trace legend label.

    Returns
    -------
    figure, axis, result
        Matplotlib objects and AnnotationResult (artists, legend handles, notes).
        Display or save the returned figure in the calling cell.
    """
    #=== 1| Validate the supplied samples ========
    time = np.asarray(time, dtype=float)
    values = np.asarray(values, dtype=float)
    if time.ndim != 1 or values.shape != time.shape or len(time) < 2:
        raise ValueError("time and values must be matching 1-D arrays with at least two samples.")
    if not np.isfinite(time).all() or not (np.diff(time) > 0).all():
        raise ValueError("time must contain finite increasing seconds.")

    #=== 2| Draw the trace and annotations using the same display offset ========
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(time - time_offset, values, color="0.2", linewidth=1, label=trace_label)
    ax.set(
        xlabel="Session time (s)" if time_offset == 0 else "Time from chosen origin (s)",
        ylabel=ylabel,
        title=title,
    )
    if ylim is not None:
        ax.set_ylim(*ylim)
    result = draw_annotations(
        ax, annotations, window=window, time_offset=time_offset, legend=False,
    )

    #=== 3| Place the key outside the trace and return the figure ========
    if result.legend_handles:
        ax.legend(handles=result.legend_handles, loc="upper center",
                  bbox_to_anchor=(0.5, -0.2), ncol=4, fontsize=8)
    fig.tight_layout()
    return fig, ax, result

## Call movement and annotate one selected trial at a time

Speed is calculated on the full recording before trial slicing. The exact event IDs prevent unrelated equal-time events from appearing in a trial’s annotation input.

In [ ]:
# === 1| Measure Each Full Recording Before Selecting Trials ======================

annotated_figures = {}
for session_id, pose in pose_by_session.items():
    point = pose.position.sel(keypoint=TRACKING_KEYPOINT, individual='individual_0')
    if point.sizes["time"] < 2:
        print(f"{session_id}: fewer than two pose samples; speed cannot be calculated.")
        continue

    speed = compute_speed(point)                         # movement differentiates position using its acquired time coordinates.
    speed = speed.where(point.notnull().all("space"))     # A missing tracked position cannot supply a valid speed at that sample.
    recording_trials = slice_stream(selected_trials, selectors={'session': session_id})

    # === 2| Pass One Trial's Explicit Data to the Existing Plot Function ==========

    for _, trial in recording_trials.iterrows():
        trial_speed = slice_stream_for_trial(speed, trial, time_coord='time')
        if trial_speed.sizes['time'] < 2:
            print(f'{session_id}: trial {int(trial["trial_index"])} has fewer than two pose samples; omitted.')
            continue
        trial_events = slice_stream_for_trial(streams['events'], trial)
        trial_table = trial.to_frame().T                 # The existing annotation adapter accepts a one-row trial DataFrame.
        annotations = qc_annotations(trial_table, trial_events)

        key = (session_id, int(trial['trial_index']))
        annotated_figures[key] = plot_annotated_trace(
            trial_speed.time.values,
            trial_speed.values,
            annotations,
            window=(float(trial['start_time']), float(trial['end_time'])),
            time_offset=float(trial['start_time']),       # Display seconds since this trial's start while retaining absolute annotation times.
            ylabel='Speed (pixels/s)',
            title=f'{session_id} — trial {key[1]}',
        )
        plt.show()